In [1]:
!pip install rdflib
!pip install urllib
!pip install pyvis openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 615.4/615.4 kB 13.2 MB/s  0:00:00
ERROR: Could not find a version that satisfies the requirement urllib (from versions: none)
ERROR: No matching distribution found for urllib
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 12.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 36.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [pyvis]


In [ ]:
from rdflib import Graph, Namespace, Literal, URIRef
from rdflib.namespace import RDF, RDFS, XSD
import pandas as pd
from urllib.parse import quote  # Added for URI encoding


# Load Excel data (assuming pandas can read your file)
df = pd.read_excel("/content/Cleaned_Online_Retail.xlsx")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# reading the dataset
df.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850,United Kingdom


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 392732 entries, 0 to 392731
Data columns (total 8 columns):
 #   Column       Non-Null Count   Dtype         
---  ------       --------------   -----         
 0   InvoiceNo    392732 non-null  int64         
 1   StockCode    392732 non-null  object        
 2   Description  392732 non-null  object        
 3   Quantity     392732 non-null  int64         
 4   InvoiceDate  392732 non-null  datetime64[ns]
 5   UnitPrice    392732 non-null  float64       
 6   CustomerID   392732 non-null  int64         
 7   Country      392732 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(3), object(3)
memory usage: 24.0+ MB


In [ ]:
l=set(list(df.CustomerID))
len(l)

4339

In [ ]:
l

In [ ]:
print(df.isnull().sum())

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


In [ ]:
#preprocessing
# dropping cancelations and null customers
df = df[~  ((df["InvoiceNo"].str.startswith("C")) | (df["CustomerID"].isna())) ]

# dropping the duplicate rows
df.drop_duplicates(inplace=True)


# Removing negative quantities

df = df_clean[df_clean['Quantity'] > 0]


AttributeError: Can only use .str accessor with string values!

In [ ]:
print(df.isnull().sum())


InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
CustomerID     0
Country        0
dtype: int64


In [ ]:
from rdflib import Graph, Namespace, Literal, URIRef
from rdflib.namespace import RDF, XSD
import pandas as pd

# Define namespaces
EX = Namespace("http://example.org/OnlineRetail#")
SCHEMA = Namespace("http://schema.org/")
XS = Namespace("http://www.w3.org/2001/XMLSchema#")

def sanitize_uri_component(value):
    """Replace invalid URI characters with underscores"""
    return str(value).strip().replace(" ", "_").replace("/", "-").replace(",", "")

def xlsx_to_rdf(input_xlsx, output_ttl='online_retail.ttl'):
    # Create RDF graph
    g = Graph()
    g.bind("ex", EX)
    g.bind("xsd", XS)
    g.bind("schema", SCHEMA)

    # Read Excel file
    df = pd.read_excel(input_xlsx, engine='openpyxl')

    for _, row in df.iterrows():
        # Skip rows with missing essential data
        if pd.isna(row.get('InvoiceNo')) or pd.isna(row.get('StockCode')):
            continue

        # Sanitize and create URIs
        invoice_no = sanitize_uri_component(row['InvoiceNo'])
        stock_code = sanitize_uri_component(row['StockCode'])

        invoice_uri = URIRef(EX[f"Invoice_{invoice_no}"])
        product_uri = URIRef(EX[f"Product_{stock_code}"])
        invoice_line_uri = URIRef(EX[f"InvoiceLine_{invoice_no}_{stock_code}"])

        # Add Invoice
        g.add((invoice_uri, RDF.type, EX.Invoice))
        g.add((invoice_uri, EX.invoiceID, Literal(invoice_no)))

        # Add InvoiceDate
        if not pd.isna(row.get('InvoiceDate')):
            try:
                dt = row['InvoiceDate'].to_pydatetime().isoformat()
                g.add((invoice_uri, EX.invoiceDate, Literal(dt, datatype=XSD.dateTime)))
            except AttributeError:
                pass

        # Add Product
        g.add((product_uri, RDF.type, EX.Product))
        g.add((product_uri, EX.stockCode, Literal(stock_code)))

        if not pd.isna(row.get('Description')):
            g.add((product_uri, EX.description, Literal(str(row['Description']).strip())))

        if not pd.isna(row.get('UnitPrice')):
            try:
                g.add((product_uri, EX.unitPrice, Literal(float(row['UnitPrice']), datatype=XSD.decimal)))
            except ValueError:
                pass

        # Add Person
        if not pd.isna(row.get('CustomerID')):
            customer_id = str(row['CustomerID']).strip()
            person_uri = URIRef(EX[f"Person_{customer_id}"])

            g.add((person_uri, RDF.type, EX.Person))
            g.add((person_uri, EX.personID, Literal(customer_id)))

            if not pd.isna(row.get('Country')):
                g.add((person_uri, EX.country, Literal(str(row['Country']).strip())))

            g.add((invoice_uri, EX.customer, person_uri))

        # Add InvoiceLine
        g.add((invoice_line_uri, RDF.type, EX.InvoiceLine))

        if not pd.isna(row.get('Quantity')):
            try:
                g.add((invoice_line_uri, EX.quantity, Literal(int(row['Quantity']), datatype=XSD.integer)))
            except ValueError:
                pass

        g.add((invoice_line_uri, EX.refersToProduct, product_uri))
        g.add((invoice_line_uri, EX.isPartOfInvoice, invoice_uri))
        g.add((invoice_uri, EX.hasInvoiceLine, invoice_line_uri))

    # Save the graph
    g.serialize(destination=output_ttl, format='turtle')
    print(f"Knowledge graph saved to {output_ttl}")



# Usage
xlsx_to_rdf('/content/Cleaned_Online_Retail.xlsx')

Knowledge graph saved to online_retail.ttl


In [ ]:
from rdflib import Graph

# Load the TTL file
g = Graph()
g.parse("online_retail.ttl", format="turtle")

# Print summary information
print("Total number of triples:", len(g))

# List distinct namespaces
print("Namespaces in the graph:")
for prefix, namespace in g.namespaces():
    print(f"{prefix}: {namespace}")

# Get distinct subjects, predicates, and objects counts
subjects = set(g.subjects())
predicates = set(g.predicates())
objects = set(g.objects())

print("Number of distinct subjects:", len(subjects))
print("Number of distinct predicates:", len(predicates))
print("Number of distinct objects:", len(objects))

# Optional: Execute a SPARQL query to list all classes (rdf:type statements)
query = """
    SELECT DISTINCT ?class
    WHERE { ?s rdf:type ?class }
"""
for row in g.query(query):
    print("Class:", row[0])


Total number of triples: 2051215
Namespaces in the graph:
brick: https://brickschema.org/schema/Brick#
csvw: http://www.w3.org/ns/csvw#
dc: http://purl.org/dc/elements/1.1/
dcat: http://www.w3.org/ns/dcat#
dcmitype: http://purl.org/dc/dcmitype/
dcterms: http://purl.org/dc/terms/
dcam: http://purl.org/dc/dcam/
doap: http://usefulinc.com/ns/doap#
foaf: http://xmlns.com/foaf/0.1/
geo: http://www.opengis.net/ont/geosparql#
odrl: http://www.w3.org/ns/odrl/2/
org: http://www.w3.org/ns/org#
prof: http://www.w3.org/ns/dx/prof/
prov: http://www.w3.org/ns/prov#
qb: http://purl.org/linked-data/cube#
schema: https://schema.org/
sh: http://www.w3.org/ns/shacl#
skos: http://www.w3.org/2004/02/skos/core#
sosa: http://www.w3.org/ns/sosa/
ssn: http://www.w3.org/ns/ssn/
time: http://www.w3.org/2006/time#
vann: http://purl.org/vocab/vann/
void: http://rdfs.org/ns/void#
wgs: https://www.w3.org/2003/01/geo/wgs84_pos#
owl: http://www.w3.org/2002/07/owl#
rdf: http://www.w3.org/1999/02/22-rdf-syntax-ns#
rdfs:

In [ ]:
#Extract Person Entities with Full Context
from rdflib import Graph, URIRef

# Namespace setup
ns = {"ex": "http://example.org/OnlineRetail#"}
person_class = URIRef(ns["ex"] + "Person")

# Get all unique persons
person_uris = set(g.subjects(predicate=URIRef("http://www.w3.org/1999/02/22-rdf-syntax-ns#type"),
                           object=person_class))

In [ ]:
def get_person_profile(person_uri):
    person_id = g.value(person_uri, EX.personID)
    country = g.value(person_uri, EX.country)

    profile = f"Customer {person_id} from {country} purchased:\n"

    # Get all invoices linked to this person
    invoices = g.subjects(EX.customer, person_uri)

    for invoice in invoices:
        # Get all invoice lines in each invoice
        invoice_lines = g.objects(invoice, EX.hasInvoiceLine)
        for line in invoice_lines:
            product = g.value(line, EX.refersToProduct)
            quantity = g.value(line, EX.quantity)
            description = g.value(product, EX.description)

            if description:  # Skip if no description
                try:
                    quantity = int(str(quantity))
                    profile += f"- {quantity}x {description}\n"
                except (ValueError, TypeError):
                    profile += f"- {description}\n"

    return profile


In [ ]:
person_uris


{rdflib.term.URIRef('http://example.org/OnlineRetail#Person_14603'),
 rdflib.term.URIRef('http://example.org/OnlineRetail#Person_13016'),
 rdflib.term.URIRef('http://example.org/OnlineRetail#Person_15584'),
 rdflib.term.URIRef('http://example.org/OnlineRetail#Person_14754'),
 rdflib.term.URIRef('http://example.org/OnlineRetail#Person_15471'),
 rdflib.term.URIRef('http://example.org/OnlineRetail#Person_16761'),
 rdflib.term.URIRef('http://example.org/OnlineRetail#Person_16670'),
 rdflib.term.URIRef('http://example.org/OnlineRetail#Person_16705'),
 rdflib.term.URIRef('http://example.org/OnlineRetail#Person_14697'),
 rdflib.term.URIRef('http://example.org/OnlineRetail#Person_13121'),
 rdflib.term.URIRef('http://example.org/OnlineRetail#Person_13068'),
 rdflib.term.URIRef('http://example.org/OnlineRetail#Person_17893'),
 rdflib.term.URIRef('http://example.org/OnlineRetail#Person_14641'),
 rdflib.term.URIRef('http://example.org/OnlineRetail#Person_13006'),
 rdflib.term.URIRef('http://exampl

In [ ]:
len(person_uris)

4339

In [ ]:

from rdflib import URIRef
from rdflib import Graph, Namespace, Literal, URIRef
from rdflib.namespace import RDF, XSD
# Define namespaces
EX = Namespace("http://example.org/OnlineRetail#")
SCHEMA = Namespace("http://schema.org/")
XS = Namespace("http://www.w3.org/2001/XMLSchema#")
# Construis l’objet URIRef à partir de la chaîne
person_uri = URIRef("http://example.org/OnlineRetail#Person_12553")

# Appelle la fonction avec l’URIRef, pas une chaîne
profile = get_person_profile(person_uri)

print(profile)


Customer 12553 from France purchased:
- 15x RETRO PLASTIC POLKA TRAY
- 24x RED TOADSTOOL LED NIGHT LIGHT
- 10x CHARLOTTE BAG SUKI DESIGN
- 10x JUMBO BAG SPACEBOY DESIGN
- 12x LIPSTICK PEN RED
- 12x SET OF 9 HEART SHAPED BALLOONS
- 48x RABBIT NIGHT LIGHT
- 8x DOLLY GIRL MINI BACKPACK
- 24x FEATHER PEN,LIGHT PINK
- 20x JUMBO BAG RED RETROSPOT
- 1x POSTAGE
- 2x ASSORTED COLOUR MINI CASES
- 12x PINK BLUE FELT CRAFT TRINKET BOX
- 6x STRAWBERRY LUNCH BOX WITH CUTLERY
- 12x RED TOADSTOOL LED NIGHT LIGHT
- 12x TRADITIONAL WOODEN SKIPPING ROPE
- 10x JUMBO STORAGE BAG SUKI
- 6x PAPER BUNTING RETROSPOT
- 3x BAKING SET 9 PIECE RETROSPOT
- 3x RETROSPOT TEA SET CERAMIC 11 PC
- 12x SET OF 9 HEART SHAPED BALLOONS
- 4x FELTCRAFT CUSHION OWL
- 4x FELTCRAFT CUSHION RABBIT
- 3x BAKING SET SPACEBOY DESIGN
- 4x FELTCRAFT PRINCESS CHARLOTTE DOLL
- 30x JUMBO BAG RED RETROSPOT
- 2x POSTAGE
- 3x EDWARDIAN PARASOL BLACK
- 10x JUMBO BAG WOODLAND ANIMALS
- 10x RED RETROSPOT CHARLOTTE BAG
- 10x LUNCH BAG RED RETROS

In [ ]:
text= [ get_person_profile(person) for person in person_uris]

In [ ]:
text

['Customer 14603 from United Kingdom purchased:\n- 200x ASSORTED COLOURS SILK FAN\n- 200x SWISS ROLL TOWEL PINK  SPOTS\n',
 "Customer 13016 from United Kingdom purchased:\n- 6x SPACEBOY ROCKET LOLLY MAKERS\n- 12x CHRISTMAS PUDDING TRINKET POT\n- 4x SET OF 6 SPICE TINS PANTRY DESIGN\n- 4x SWEETHEART RECIPE BOOK STAND\n- 2x BREAD BIN DINER STYLE IVORY\n- 24x FRIDGE MAGNETS US DINER ASSORTED\n- 24x FRIDGE MAGNETS LA VIE EN ROSE\n- 6x BAKING MOULD TOFFEE CUP  CHOCOLATE\n- 6x BAKING MOULD EASTER EGG WHITE CHOC\n- 6x 36 FOIL STAR CAKE CASES\n- 6x 36 FOIL HEART CAKE CASES\n- 4x BULL DOG BOTTLE OPENER\n- 6x DOILEY BISCUIT TIN\n- 6x TREASURE TIN GYMKHANA DESIGN\n- 12x BISCUIT TIN 50'S CHRISTMAS\n- 24x SET OF 12 FAIRY CAKE BAKING CASES\n- 24x SET OF 6 SNACK LOAF BAKING CASES\n- 24x SET OF 12 MINI LOAF BAKING CASES\n- 12x TRADITIONAL CHRISTMAS RIBBONS\n- 24x PACK OF 60 PINK PAISLEY CAKE CASES\n- 12x 6 RIBBONS RUSTIC CHARM\n- 24x PACK OF 60 SPACEBOY CAKE CASES\n- 1x ENAMEL BREAD BIN CREAM\n- 6x SE

In [ ]:
from transformers import BertTokenizer, BertModel
import torch
import pandas as pd
from tqdm import tqdm

# --------------------------------------------------
# 1. Prepare text profiles from Knowledge Graph
# --------------------------------------------------
# person_uris: list of KG entities
# get_person_profile(person): returns a textual profile (string)

texts = [get_person_profile(person) for person in person_uris]

# Optional: keep IDs
person_ids = [str(person).split("#")[-1] for person in person_uris]

# --------------------------------------------------
# 2. Load BERT
# --------------------------------------------------
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
model = BertModel.from_pretrained("bert-base-uncased")
model.eval()

# Move to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# --------------------------------------------------
# 3. Generate BERT embeddings
# --------------------------------------------------
embeddings = []

for text in tqdm(texts, desc="Encoding customer profiles"):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=256
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

        # [CLS] token embedding (recommended for clustering)
        cls_embedding = outputs.last_hidden_state[:, 0, :]  # shape: [1, 768]
        embeddings.append(cls_embedding.cpu().numpy().flatten())

# --------------------------------------------------
# 4. Convert to DataFrame
# --------------------------------------------------
bert_embeddings = pd.DataFrame(embeddings)
bert_embeddings["PersonID"] = person_ids

# --------------------------------------------------
# 5. Save embeddings
# --------------------------------------------------
bert_embeddings.to_csv("bert_embeddings_from_KG_profiles.csv", index=False)

print("✅ BERT embeddings extracted from KG textual profiles and saved.")


Encoding customer profiles: 100%|██████████| 4339/4339 [20:30<00:00,  3.53it/s]


✅ BERT embeddings extracted from KG textual profiles and saved.


In [ ]:
import pandas as pd

df = pd.read_csv("bert_embeddings_from_KG_profiles.csv")

customer_ids = df["PersonID"].values
embeddings = df.drop(columns=["PersonID"]).values